[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/charlesincharge/Caltech-CS155-2022/blob/main/sets/set5/set5_prob3.ipynb)


## Set 5
## 3. Word2Vec \*\*Principles**

#### Preparation

In [3]:
# download the helper function
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/charlesincharge/Caltech-CS155-2022/main/sets/set5/P3CHelpers.py', 'P3CHelpers.py')

('P3CHelpers.py', <http.client.HTTPMessage at 0x1f968ca3c10>)

In [4]:
# download the dataset
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/charlesincharge/Caltech-CS155-2022/main/sets/set5/data/dr_seuss.txt', 'dr_seuss.txt')

('dr_seuss.txt', <http.client.HTTPMessage at 0x1f968ca3460>)

In [5]:
import numpy as np
from P3CHelpers import *
import torch
import torch.nn as nn
import torch.optim as optim

#### Problem D: 
Fill in the generate_traindata and find_most_similar_pairs functions.

In [6]:
def get_word_repr(word_to_index, word):
    """
    Returns one-hot-encoded feature representation of the specified word given
    a dictionary mapping words to their one-hot-encoded index.

    Arguments:
        word_to_index: Dictionary mapping words to their corresponding index
                       in a one-hot-encoded representation of our corpus.

        word:          Word whose feature representation we wish to compute.

    Returns:
        feature_representation:     Feature representation of the passed-in word.
    """
    unique_words = word_to_index.keys()
    # Return a vector that's zero everywhere besides the index corresponding to <word>
    feature_representation = np.zeros(len(unique_words))
    feature_representation[word_to_index[word]] = 1
    return feature_representation    

def generate_traindata(word_list, word_to_index, window_size=4):
    """
    Generates training data for Skipgram model.

    Arguments:
        word_list:     Sequential list of words (strings).
        word_to_index: Dictionary mapping words to their corresponding index
                       in a one-hot-encoded representation of our corpus.

        window_size:   Size of Skipgram window. Defaults to 2 
                       (use the default value when running your code).

    Returns:
        (trainX, trainY):     A pair of matrices (trainX, trainY) containing training 
                              points (one-hot-encoded vectors) and their corresponding output_word
                              (also one-hot-encoded vectors)

    """
    trainX = []
    trainY = []

    ##############################################################
    # TODO: Implement this function, populating trainX and trainY
    ##############################################################
    for i in range(len(word_list)):
        word = word_list[i]
        for j in range(max(0, i - window_size), min(len(word_list), i + window_size + 1)):
            if j != i:
                trainX.append(get_word_repr(word_to_index, word))
                trainY.append(get_word_repr(word_to_index, word_list[j]))
    
    return np.array(trainX), np.array(trainY)

In [10]:
def find_most_similar_pairs(filename, num_latent_factors):
    """
    Find the most similar pairs from the word embeddings computed from
    a body of text
    
    Arguments:
        filename:           Text file to read and train embeddings from
        num_latent_factors: The number of latent factors / the size of the embedding
    """
    # Load in a list of words from the specified file; remove non-alphanumeric characters
    # and make all chars lowercase.
    sample_text = load_word_list(filename)

    # Create word dictionary
    word_to_index = generate_onehot_dict(sample_text)
    print("Textfile contains %s unique words"%len(word_to_index))
    # Create training data
    trainX, trainY = generate_traindata(sample_text, word_to_index)

    ##############################################################
    # TODO: 1) Create and train model in Pytorch.      
    ##############################################################

    # vocab_size = number of unique words in our text file. Will be useful 
    # when adding layers to your neural network
    vocab_size = len(word_to_index)

    # Create a simple neural network model (no activation — standard Word2Vec)
    model = nn.Sequential(
        nn.Linear(vocab_size, num_latent_factors),
        nn.Linear(num_latent_factors, vocab_size)
    )
    
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Convert training data to tensors
    trainX_tensor = torch.tensor(trainX, dtype=torch.float32)
    trainY_tensor = torch.tensor(np.argmax(trainY, axis=1), dtype=torch.long)

    # Train the model
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(trainX_tensor)
        loss = criterion(outputs, trainY_tensor)
        loss.backward()
        optimizer.step()

    ##############################################################
    # TODO: 2) Extract weights for hidden layer
    ##############################################################
    
    # set weights variable below (transpose so each row is a word embedding)
    weights = model[0].weight.data.numpy().T
    
    # Find and print most similar pairs
    similar_pairs = most_similar_pairs(weights, word_to_index)
    for pair in similar_pairs[:30]:
        print(pair)

### Problem E-H:
Run your model on drseuss.txt and answer questions from E through H.

In [ ]:
find_most_similar_pairs('dr_seuss.txt', 10)

Textfile contains 308 unique words
Pair(yet, met), Similarity: 0.97927403
Pair(met, yet), Similarity: 0.97927403
Pair(every, day), Similarity: 0.9784291
Pair(day, every), Similarity: 0.9784291
Pair(wump, hump), Similarity: 0.97204185
Pair(hump, wump), Similarity: 0.97204185
Pair(today, gone), Similarity: 0.9693673
Pair(gone, today), Similarity: 0.9693673
Pair(off, foot), Similarity: 0.9605223
Pair(foot, off), Similarity: 0.9605223
Pair(dish, help), Similarity: 0.95601225
Pair(help, dish), Similarity: 0.95601225
Pair(too, bump), Similarity: 0.95186615
Pair(bump, too), Similarity: 0.95186615
Pair(finger, top), Similarity: 0.9517245
Pair(top, finger), Similarity: 0.9517245
Pair(ear, fear), Similarity: 0.95156443
Pair(fear, ear), Similarity: 0.95156443
Pair(goat, boat), Similarity: 0.9508201
Pair(boat, goat), Similarity: 0.9508201
Pair(play, game), Similarity: 0.94839406
Pair(game, play), Similarity: 0.94839406
Pair(back, play), Similarity: 0.94810104
Pair(there, here), Similarity: 0.94804

: 